# Active learning for molecules, starting from zero Python

This notebook is for people who **don't know much Python**. You'll learn just enough
Python to understand what the `al-molecular` project does, using one running example
the whole way through:

> We have many molecules. A computer program (docking) can tell us how well each one
> sticks to a protein, but it's slow. **How do we find the best molecules while running
> it as few times as possible?**

### How to use a notebook

* A notebook is made of **cells**. Grey boxes are **code**; text like this is a **note**.
* Click a code cell and press **Shift + Enter** to run it. Its result appears just below.
* Run the cells **in order, from top to bottom**. Later cells use things made in earlier ones.
* If something gets confusing, use the menu to **Restart** the kernel and run everything again from the top.
* You can't break anything. Change numbers, rerun, and see what happens. That's how you learn.

In VS Code, pick the **py310** kernel (top right) before you start.

---
## Part 1: Values and variables

A **variable** is a name for a value. You create one with `=`, which you can read as
"is set to" (it does *not* mean "is equal to" like in maths).

Anything after a `#` is a **comment**. Python ignores it; it's a note for humans.

`print(...)` shows something on the screen.

In [ ]:
score = -2.6          # a docking score. In docking, LOWER (more negative) = sticks BETTER
print(score)

Text goes in quotes and is called a **string**. Molecules are often written as text
in a format called **SMILES**. For example, `CCO` is ethanol: two carbons (C) and one oxygen (O).

In [ ]:
smiles = "CCO"
name = "ethanol"
print(name, "is written as", smiles)

An **f-string** (an `f` before the quotes) lets you put variables inside text using `{ }`.
You'll see these everywhere in al-molecular's printouts.

In [ ]:
print(f"{name} ({smiles}) has a docking score of {score}")

You can do arithmetic. `*` means multiply and `/` means divide.

In [ ]:
docked = 600
pool_size = 5000
print(docked / pool_size)                 # fraction of the pool we tested
print(f"{docked / pool_size:.0%}")        # same number, shown as a percent (the :.0% part)

**Your turn:** change `docked` to `50` in the cell above and run it again.

**What you learned:** variables (`=`), numbers, strings (text in quotes), `print`, f-strings.

---
## Part 2: Lists, for keeping things in order

A **list** holds several values in order, written inside square brackets `[ ]`.

In [ ]:
molecules = ["CCO", "c1ccccc1", "CC(=O)C"]     # ethanol, benzene, acetone
print(molecules)
print(len(molecules))                           # len = how many items

To get one item, use its **position** in square brackets. Python starts counting at **0**,
so the first item is `[0]`. This catches everyone at first.

In [ ]:
print(molecules[0])     # first
print(molecules[1])     # second
print(molecules[-1])    # -1 means "the last one"

`.append(...)` adds an item to the end. A **slice** `[start:stop]` takes a piece of the list (it stops *before* `stop`).

In [ ]:
molecules.append("Cc1ccccc1")     # toluene
print(molecules)
print(molecules[0:2])             # items 0 and 1, i.e. the first two

**What you learned:** lists `[ ]`, counting from 0, `len`, `.append`, slices `[a:b]`.

---
## Part 3: Dictionaries, for looking things up

A **dictionary** stores **pairs**: a *key* and its *value*, written `key: value` inside
curly brackets `{ }`. Think of a real dictionary: you look up a word (key) and get its
meaning (value).

In al-molecular, the most important dictionary is the **oracle**: molecule → docking
score. Here's a tiny one with 13 real molecules. **The scores are made up for this
lesson**, but like real docking scores, bigger molecules tend to score better (more negative).

In [ ]:
oracle = {
    "CCO": -2.6,                            # ethanol
    "CC(=O)C": -3.5,                        # acetone
    "c1ccccc1": -4.1,                       # benzene
    "Cc1ccccc1": -5.1,                      # toluene
    "Oc1ccccc1": -4.2,                      # phenol
    "c1ccc2ccccc2c1": -6.3,                 # naphthalene
    "CC(=O)Nc1ccc(O)cc1": -6.2,             # paracetamol
    "CN1CCCC1c1cccnc1": -7.5,               # nicotine
    "OCC1OC(O)C(O)C(O)C1O": -5.7,           # glucose
    "CC(=O)Oc1ccccc1C(=O)O": -7.4,          # aspirin
    "NCCc1c[nH]c2ccc(O)cc12": -8.3,         # serotonin
    "Cn1cnc2c1c(=O)n(C)c(=O)n2C": -7.4,     # caffeine
    "CC(C)Cc1ccc(cc1)C(C)C(=O)O": -8.3,     # ibuprofen
}

names = {
    "CCO": "ethanol",
    "CC(=O)C": "acetone",
    "c1ccccc1": "benzene",
    "Cc1ccccc1": "toluene",
    "Oc1ccccc1": "phenol",
    "c1ccc2ccccc2c1": "naphthalene",
    "CC(=O)Nc1ccc(O)cc1": "paracetamol",
    "CN1CCCC1c1cccnc1": "nicotine",
    "OCC1OC(O)C(O)C(O)C1O": "glucose",
    "CC(=O)Oc1ccccc1C(=O)O": "aspirin",
    "NCCc1c[nH]c2ccc(O)cc12": "serotonin",
    "Cn1cnc2c1c(=O)n(C)c(=O)n2C": "caffeine",
    "CC(C)Cc1ccc(cc1)C(C)C(=O)O": "ibuprofen",
}

print(len(oracle), "molecules")

Look up a value by putting the key in square brackets:

In [ ]:
print(oracle["CCO"])
print(names["CCO"], "scores", oracle["CCO"])

If you ask for a key that isn't there, Python stops with an **error** (a `KeyError`).
**Errors are normal.** Read the *last line* of the error message; it usually tells you
what went wrong. Remove the `#` from the line below and run it to see one, then put the `#` back.

In [ ]:
# oracle["XYZ"]

You can check whether a key is there with `in`, and add a new pair with `=`:

In [ ]:
print("CCO" in oracle)          # True
print("XYZ" in oracle)          # False

my_results = {}                 # an EMPTY dictionary
my_results["CCO"] = -2.6        # add a pair
print(my_results)

In al-molecular, the real oracle is loaded from a file by `load_oracle()` and holds about
2 million pairs. Exactly the same idea, just bigger.

**What you learned:** dictionaries `{key: value}`, looking up with `[key]`, `in`, adding pairs, and reading an error.

---
## Part 4: Loops and decisions

A **for loop** does something once for every item. The lines that belong to the loop
are **indented** (pushed right by 4 spaces). In Python, indentation is not decoration:
it's how Python knows which lines are inside the loop.

In [ ]:
for smi in molecules:
    print("looking at", smi)
print("done")      # not indented, so this runs once, after the loop

Looping over a dictionary gives you its keys. Then use the key to get the value:

In [ ]:
for smi in oracle:
    print(f"{names[smi]:12s} {oracle[smi]}")      # :12s pads the name to 12 characters so it lines up

`if` makes a decision. `<` means "less than". Lines under the `if` only run when the
condition is true; `else` covers the other case.

In [ ]:
for smi in oracle:
    if oracle[smi] < -6:
        print(names[smi], "-> good binder")
    else:
        print(names[smi], "-> weak")

A loop can also **count** or **add up**. Here's a useful one: roughly how big is a
molecule? Count the letters in its SMILES (each letter is roughly one atom).
`ch.isalpha()` is `True` when `ch` is a letter. `+= 1` means "add one".

In [ ]:
def size(smiles):
    count = 0
    for ch in smiles:          # a string is a sequence of characters, so you can loop over it
        if ch.isalpha():
            count += 1
    return count

print(size("CCO"), size("c1ccccc1"))

Oops, we just used something new: `def`. That's the next part.

**What you learned:** `for` loops, indentation, `if` / `else`, counting with `+=`.

---
## Part 5: Functions, for naming a recipe

`def name(inputs):` creates a **function**: a named recipe you can reuse. `return`
hands back the answer. You already wrote `size(...)` above. Here's another one:

In [ ]:
def dock(smiles):
    # In real life this would run a slow docking program.
    # Here we just look the answer up in our oracle.
    return oracle[smiles]

print(dock("CC(=O)Oc1ccccc1C(=O)O"))     # aspirin

**Your turn:** use `size` and `dock` together to print each molecule's name, size and score.

In [ ]:
for smi in oracle:
    print(f"{names[smi]:12s} size={size(smi):2d}  score={dock(smi)}")

**What you learned:** `def`, inputs, `return`, and calling a function by name with `( )`.

---
## Part 6: Sorting, and finding the best

`sorted(...)` puts things in order. The `key=` part says **what to sort by**. Here we
sort the molecules by their docking score, lowest (best) first.

In [ ]:
ranked = sorted(oracle, key=dock)       # "sort the molecules, using dock(molecule) as the value"
for smi in ranked:
    print(names[smi], dock(smi))

The **true top 3** are the first three in that list. We'll pretend we *don't know*
this list (in real life, knowing it would mean docking everything) and only use it at
the end to grade ourselves.

In [ ]:
TOPK = 3
true_top = ranked[:TOPK]
print("the answer key:", [names[s] for s in true_top])

That last line used a **list comprehension**: `[names[s] for s in true_top]` means
"make a list of `names[s]` for each `s` in `true_top`". It's a short way to write a
loop that builds a list, and Python code uses it a lot.

**What you learned:** `sorted` with `key=`, taking the first few with `[:3]`, and list comprehensions.

---
## Part 7: Active learning, by hand, on 13 molecules

Now the real idea. The rules of the game:

1. We may **dock only a few molecules**. Every `dock(...)` call costs money and time.
2. Start by docking a few **at random**.
3. Use what we've learned to **guess** the scores of the rest.
4. Dock the ones with the **best guesses**, then repeat.

This is **active learning**, and it's what al-molecular does with 2 million molecules.

**Step 1: dock 3 at random.** `random.sample(list, 3)` picks 3 different items.
`random.seed(...)` makes the "random" choice the same every time you run the notebook.

In [ ]:
import random            # "import" loads a toolbox that comes with Python
random.seed(7)

pool = list(oracle)                      # all 13 molecules
docked = {}                              # what we've learned so far: {smiles: score}

for smi in random.sample(pool, 3):
    docked[smi] = dock(smi)

for smi in docked:
    print(f"docked {names[smi]:12s} size={size(smi):2d}  score={docked[smi]}")

**Step 2: guess the scores of the others.** We need a rule that guesses a score
without docking. This rule is called the **surrogate**. Ours is very simple:

> "A molecule probably scores like the docked molecule closest to it in size."

`abs(x)` means "x without its minus sign", so `abs(size(a) - size(b))` is how
different the two sizes are.

In [ ]:
def guess(smi):
    closest = None
    for known in docked:
        if closest is None or abs(size(known) - size(smi)) < abs(size(closest) - size(smi)):
            closest = known
    return docked[closest]

undocked = [smi for smi in pool if smi not in docked]
for smi in undocked:
    print(f"{names[smi]:12s} size={size(smi):2d}  guess={guess(smi)}")

**Step 3: pick the 2 best-looking ones** (lowest guess) and dock them for real.

In [ ]:
BATCH = 2
picks = sorted(undocked, key=guess)[:BATCH]       # same sorted() trick as Part 6
for smi in picks:                                 # first, show what we chose and why
    print(f"picked {names[smi]:12s}  guess {guess(smi)}")

for smi in picks:                                 # then dock the whole batch
    docked[smi] = dock(smi)
    print(f"docked {names[smi]:12s}  real score {docked[smi]}")

Look at glucose: its size made it look promising, but its real score is weak. A guess
is only a guess. That's why we dock the picks for real and **learn from the result**:
next round, the guesses will know about glucose.

**Step 4: how are we doing?** **Recall** = the fraction of the true top 3 that we've
docked so far. al-molecular reports exactly this number (for the top 1000).

In [ ]:
def recall():
    found = 0
    for smi in true_top:
        if smi in docked:
            found += 1
    return found / TOPK

print(f"docked {len(docked)} of {len(pool)}, recall = {recall():.0%}")

**Repeat.** Run the three cells "Step 2", "Step 3" and "Step 4" again (each round, the
guesses use everything docked so far). Do it twice and watch the recall.

Here is the same thing written as one loop, starting fresh:

In [ ]:
random.seed(7)
docked = {}
for smi in random.sample(pool, 3):
    docked[smi] = dock(smi)
print(f"start:   docked {len(docked)}, recall = {recall():.0%}")

for round_number in range(1, 4):          # range(1, 4) gives 1, 2, 3
    undocked = [smi for smi in pool if smi not in docked]
    for smi in sorted(undocked, key=guess)[:BATCH]:
        docked[smi] = dock(smi)
    print(f"round {round_number}: docked {len(docked)}, recall = {recall():.0%}")

With 9 of the 13 molecules docked, we found all of the top 3. Docking at random would
need about 10 to 11 tries on average to be sure of getting all three. On 13 molecules
that's a small win, but on 2 million molecules the gap becomes enormous, as you'll see next.

**Notice:** our "size" rule only works because size really does matter for these
scores. In the real project, choosing what the surrogate can "see" about a molecule
turned out to be the most important decision of all.

---
## Part 8: NumPy, for doing the same thing to thousands of numbers at once

Plain Python loops are fine for 13 molecules but slow for millions. **NumPy** is a
toolbox for big tables of numbers (called **arrays**). Its key idea: you write the
operation **once** and it is applied to every number.

In [ ]:
import numpy as np          # "as np" = a short nickname everyone uses

scores = np.array([-9.2, -5.1, -7.7, -11.0])
print(scores * 2)           # every number doubled; no loop needed
print(scores.min(), scores.mean())
print(scores < -7)          # a True/False for each number

Three NumPy tricks al-molecular uses constantly:

* `np.argsort(x)` gives the **positions** that would sort `x`. So `np.argsort(x)[:2]` is "where are the 2 smallest?".
* `x[positions]` picks out those items.
* `x[x < -7]` keeps only the items where the condition is True (a **mask**).

In [ ]:
order = np.argsort(scores)
print("positions, best first:", order)
print("best two scores:      ", scores[order[:2]])
print("only the good ones:   ", scores[scores < -7])

Now a **pretend pool of 5,000 molecules**. Each one is described by 8 numbers instead
of a SMILES string. In al-molecular these numbers come from big pretrained AI models
and are called **embeddings**. The docking score depends on those numbers in a way the
algorithm doesn't know.

`rng.normal(size=(5000, 8))` makes a table of random numbers with 5000 rows and 8 columns.

In [ ]:
rng = np.random.default_rng(0)
N_POOL = 5000
embeddings = rng.normal(size=(N_POOL, 8))
e = embeddings
true_scores = -8 + 1.2 * e[:, 0] - 0.8 * e[:, 1] + 0.6 * e[:, 2] * e[:, 3] + rng.normal(scale=0.4, size=N_POOL)
# e[:, 0] means "column 0, all rows"

print("table shape:", embeddings.shape)
print("first molecule's 8 numbers:", embeddings[0].round(2))
print("first 5 scores:", true_scores[:5].round(2))

The surrogate this time is a "best straight-line rule" (linear regression). **You don't
need the maths.** Just know that `fit_rule` learns from docked molecules, and
`use_rule` produces guesses for any molecules. These two steps (*fit* and *predict*) are
exactly what every surrogate in al-molecular's `surrogates.py` does.

In [ ]:
def fit_rule(X, y):
    X1 = np.c_[X, np.ones(len(X))]
    return np.linalg.lstsq(X1, y, rcond=None)[0]

def use_rule(rule, X):
    return np.c_[X, np.ones(len(X))] @ rule

---
## Part 9: The full loop, AI-guided vs random

The same four steps as Part 7, now with NumPy and 5,000 molecules. We'll try two
strategies with the **same budget**:

* `"random"`: just dock random molecules (the baseline)
* `"guided"`: dock the molecules the rule thinks are best (active learning)

We grade against the true **top 50**.

In [ ]:
TOP50 = np.argsort(true_scores)[:50]          # positions of the 50 best (lowest) scores

def run(strategy, rounds=5, start=100, batch=100, seed=42):
    r = np.random.default_rng(seed)
    is_docked = np.zeros(N_POOL, dtype=bool)                  # one True/False per molecule
    is_docked[r.choice(N_POOL, start, replace=False)] = True  # dock a random start set
    history = [is_docked[TOP50].mean()]                       # fraction of top-50 found

    for _ in range(rounds):                                   # "_" = a loop counter we don't need
        candidates = np.where(~is_docked)[0]                  # positions NOT docked yet (~ means "not")
        if strategy == "random":
            picks = r.choice(candidates, batch, replace=False)
        else:
            rule = fit_rule(embeddings[is_docked], true_scores[is_docked])   # learn from docked
            guesses = use_rule(rule, embeddings[candidates])                 # guess the rest
            picks = candidates[np.argsort(guesses)[:batch]]                  # lowest guesses
        is_docked[picks] = True                                              # "dock" them
        history.append(is_docked[TOP50].mean())
    return history

print("random:", [f"{x:.0%}" for x in run("random")])
print("guided:", [f"{x:.0%}" for x in run("guided")])

Each list shows recall at the start and after each of the 5 rounds. A picture makes the
difference obvious. **matplotlib** is the usual plotting toolbox.

In [ ]:
import matplotlib.pyplot as plt

for strategy in ["random", "guided"]:
    h = run(strategy)
    plt.plot(h, marker="o", label=f"{strategy} (final {h[-1]:.0%})")

plt.xlabel("round")
plt.ylabel("fraction of the top 50 found")
plt.title("Same budget: 600 of 5,000 molecules docked")
plt.ylim(0, 1)
plt.legend()
plt.show()

**That gap is the whole point of al-molecular.** Everything else in the project is
about making the guided curve rise even faster on real data: better descriptions of
molecules, better surrogates, better rules for picking.

**Your turn:** results depend a little on luck. Rerun with other seeds,
`run("guided", seed=1)`, `seed=2`, ... Does the guided strategy always win? The real
project runs every experiment with 5 seeds for exactly this reason.

In [ ]:
for s in [1, 2, 3]:
    print(f"seed {s}:  random {run('random', seed=s)[-1]:.0%}   guided {run('guided', seed=s)[-1]:.0%}")

---
## Part 10: Peeking at a real experiment

al-molecular saves every experiment in a folder under `runs/`. Let's read one. It used
**2.1 million real molecules** and real docking scores against a protein called
thymidylate kinase.

The file is **JSON**, a common text format for lists and dictionaries. Python's `json`
toolbox reads it into exactly the lists and dictionaries you learned above.

In [ ]:
import json
from pathlib import Path      # Path = a file location

run_folder = Path("/N/slate/mengjing/repos/al-molecular/runs/fusionfix_greedy_frac0.001_seed42")
history = json.loads((run_folder / "history.json").read_text())

print(type(history), "of", len(history), "rounds")
print(history[0])             # round 1 is a dictionary!

It's a list of dictionaries, one per round. Loop over it just like in Part 4:

In [ ]:
for rnd in history:
    print(f"round {rnd['round']}: docked {rnd['n_labeled']:,} molecules, "
          f"found {rnd['topk_recall']:.0%} of the true top 1000")

After docking about 12,600 of 2.1 million molecules (**0.6%**), it found about **78%** of
the top 1000. Random docking would have found about 0.6% of them.

Finally, the best molecules that run found. **pandas** is a toolbox for tables;
`.head(10)` shows the first 10 rows. These are real SMILES strings and real docking scores.

In [ ]:
import pandas as pd
table = pd.read_csv(run_folder / "all_explored_final.csv")
table.head(10)

---
## Summary: what you learned and where it lives in al-molecular

| Python idea | What we used it for | In al-molecular |
|---|---|---|
| variables, strings, f-strings | scores, SMILES, printouts | everywhere |
| lists, `[0]`, slices | molecules in order, "first 3" | the pool of SMILES |
| dictionaries | the oracle `{smiles: score}` | `load_oracle()`, `labeled_scores` |
| `for`, `if` | going through molecules | the main loop in `run_experiment.py` |
| functions (`def`) | `dock`, `guess`, `recall` | `_recall()`, `true_top_k_set()`, ... |
| `sorted(..., key=)` / `np.argsort` | picking the best | choosing each batch |
| NumPy arrays and masks | 5,000 molecules at once | 2 million molecules at once |
| fit / predict | the surrogate | the classes in `surrogates.py` |
| `json`, `pandas` | reading results | everything in `runs/` |

**The big idea in one sentence:** dock a few, learn a cheap guess, dock the
best-looking guesses, repeat, and you find the best molecules while testing only a tiny fraction.

**Next step:** `walkthrough.ipynb` in this same folder goes deeper: uncertainty, the
two picking rules the project compares (greedy and UCB), and the project's real AI models.